# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [1]:
from pathlib import Path
import importlib.util

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [2]:
# Parâmetros importados do arquivo config.py
print("Filtrar por quantidade de alelos TCC1:.........................", parametros.filtarar_por_qte_de_alelos_tcc1)
print("Parâmetro de filtragem median binding percentile TCC1:.........", parametros.parametro_de_filtragem_mbp_tcc1)
print("Percentual de match mínimo TCC1:...............................", parametros.percent_match_minimo_tcc1)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 5
Percentual de match mínimo TCC1:............................... 95.0


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,LSEKEENMVKSQ,338,349,12,HLA-A*23:01,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36986,1,LSEKEENMVKSQ,338,349,12,HLA-A*24:02,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36987,1,LSEKEENMVKSQ,338,349,12,HLA-A*32:01,1367,100.00,LSEKEVKSQ,LSEKEENMVKSQ,0.000000,100.00
36988,1,EKEENMVKSQVT,340,351,12,HLA-A*11:01,1369,100.00,ENMVKSQVT,EKEENMVKSQVT,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [5]:
df_mbp_m5 = df[df['median binding percentile'] < parametros.parametro_de_filtragem_mbp_tcc1].copy()
print("Filtrando por median binding percentile < ", parametros.parametro_de_filtragem_mbp_tcc1)
df_mbp_m5

Filtrando por median binding percentile <  5


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2606,1,CLWPKTHTLW,223,232,10,HLA-B*44:03,567,4.90,CLWPKTHLW,CLWPKTHTLW,0.003115,4.90
2607,1,NELNYVLWE,73,81,9,HLA-B*44:03,73,4.90,NELNYVLWE,NELNYVLWE,0.003085,4.90
2608,1,DQKAVHADMGY,190,200,11,HLA-B*44:02,877,4.90,DQKAVHMGY,DQKAVHADMGY,0.002847,4.90
2609,1,TPPVSDLKY,105,113,9,HLA-B*44:02,105,4.90,TPPVSDLKY,TPPVSDLKY,0.002830,4.90


## Agrupando por pepitideos e agregando colunas pertinentes

In [6]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAV,186,194,7,2.700,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*6..."
1,AAIKDQKAVH,186,195,2,4.200,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESSK,196,206,2,3.100,"HLA-A*03:01, HLA-A*11:01"
3,AGDVKGVLTK,90,99,3,1.400,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01"
4,AGPFSQHNY,248,256,8,2.300,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
...,...,...,...,...,...,...
593,YVLWEGGHDL,77,86,8,3.150,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*3..."
594,YVLWEGGHDLT,77,87,2,2.050,"HLA-A*02:01, HLA-A*02:06"
595,YVLWEGGHDLTV,77,88,4,2.750,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-B*4..."
596,YWIESSKNQTW,200,210,12,0.775,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."


## Filtragem por qte_de_alelos

In [7]:
filtarar_por_qte_de_alelos = 2

In [8]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= filtarar_por_qte_de_alelos
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAV,186,194,7,2.700,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*6..."
1,AAIKDQKAVH,186,195,2,4.200,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESSK,196,206,2,3.100,"HLA-A*03:01, HLA-A*11:01"
3,AGDVKGVLTK,90,99,3,1.400,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01"
4,AGPFSQHNY,248,256,8,2.300,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
...,...,...,...,...,...,...
462,YVLWEGGHDL,77,86,8,3.150,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*3..."
463,YVLWEGGHDLT,77,87,2,2.050,"HLA-A*02:01, HLA-A*02:06"
464,YVLWEGGHDLTV,77,88,4,2.750,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-B*4..."
465,YWIESSKNQTW,200,210,12,0.775,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."


## Sorting por median_biding_percentile

In [9]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,VLWEGGHDLTV,78,88,5,0.220,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*3..."
1,IFVVDNVHTW,19,28,11,0.230,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
2,VVAGDVKGV,88,96,5,0.240,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
3,SEVCDHRLM,176,184,3,0.240,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
4,RSTTRLENVMW,58,68,4,0.285,"HLA-A*32:01, HLA-B*53:01, HLA-B*57:01, HLA-B*5..."
...,...,...,...,...,...,...
462,LRFLGEDGCW,321,330,3,4.500,"HLA-A*23:01, HLA-B*57:01, HLA-B*58:01"
463,MEIRPLSEKE,333,342,2,4.700,"HLA-B*40:01, HLA-B*44:03"
464,MPPLRFLGE,318,326,3,4.700,"HLA-B*08:01, HLA-B*35:01, HLA-B*51:01"
465,QPESPARLAS,35,44,3,4.700,"HLA-B*07:02, HLA-B*44:02, HLA-B*44:03"


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [10]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0       VLWEGGHDLTV
1        IFVVDNVHTW
2         VVAGDVKGV
3         SEVCDHRLM
4       RSTTRLENVMW
           ...     
462      LRFLGEDGCW
463      MEIRPLSEKE
464       MPPLRFLGE
465      QPESPARLAS
466    QPESPARLASAI
Name: peptide, Length: 467, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [11]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [12]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,NaN
1,2,NP 2,CLWPKTHTL,9,99.33% (148/149),88.89%,100.00%,NaN
2,3,NP 3,SQMLIPKSY,9,53.69% (80/149),77.78%,100.00%,NaN
3,4,NP 4,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,NaN
4,5,NP 5,CTMPPLRFL,9,97.99% (146/149),77.78%,100.00%,NaN
...,...,...,...,...,...,...,...,...
59,60,NP 60,RLASAILNA,9,97.99% (146/149),66.67%,100.00%,NaN
60,61,NP 61,QTVGPWHLGK,10,62.42% (93/149),80.00%,100.00%,NaN
61,62,NP 62,TPPVSDLKY,9,27.52% (41/149),77.78%,100.00%,NaN
62,63,NP 63,RALTPPVSDLK,11,26.17% (39/149),63.64%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [13]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,24,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
1,NP 2,CLWPKTHTL,9,99.33% (148/149),88.89%,100.00%,23,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
2,NP 3,SQMLIPKSY,9,53.69% (80/149),77.78%,100.00%,21,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*0..."
3,NP 4,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
4,NP 5,CTMPPLRFL,9,97.99% (146/149),77.78%,100.00%,20,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
...,...,...,...,...,...,...,...,...
59,NP 60,RLASAILNA,9,97.99% (146/149),66.67%,100.00%,10,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
60,NP 61,QTVGPWHLGK,10,62.42% (93/149),80.00%,100.00%,10,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
61,NP 62,TPPVSDLKY,9,27.52% (41/149),77.78%,100.00%,10,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*0..."
62,NP 63,RALTPPVSDLK,11,26.17% (39/149),63.64%,100.00%,10,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*3..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [14]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,24,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",73.83
1,NP 2,CLWPKTHTL,9,99.33% (148/149),88.89%,100.00%,23,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",99.33
2,NP 3,SQMLIPKSY,9,53.69% (80/149),77.78%,100.00%,21,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*0...",53.69
3,NP 4,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2...",98.66
4,NP 5,CTMPPLRFL,9,97.99% (146/149),77.78%,100.00%,20,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0...",97.99
...,...,...,...,...,...,...,...,...,...
59,NP 60,RLASAILNA,9,97.99% (146/149),66.67%,100.00%,10,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0...",97.99
60,NP 61,QTVGPWHLGK,10,62.42% (93/149),80.00%,100.00%,10,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",62.42
61,NP 62,TPPVSDLKY,9,27.52% (41/149),77.78%,100.00%,10,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*0...",27.52
62,NP 63,RALTPPVSDLK,11,26.17% (39/149),63.64%,100.00%,10,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*3...",26.17


### Sort e filtragem por percent_match e presença em alelos

In [15]:
# Parametros
percent_match_minimo = 95.0

In [16]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= percent_match_minimo]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 58,AVHADMGYW,9,100.00% (149/149),100.00%,100.00%,10,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3...",100.00
1,NP 2,CLWPKTHTL,9,99.33% (148/149),88.89%,100.00%,23,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",99.33
2,NP 9,KTCLWPKTHTL,11,99.33% (148/149),90.91%,100.00%,17,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0...",99.33
3,NP 18,SQHNYRQGY,9,99.33% (148/149),88.89%,100.00%,14,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.33
4,NP 8,FQPESPARL,9,99.33% (148/149),77.78%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",99.33
5,NP 42,CLWPKTHTLW,10,99.33% (148/149),90.00%,100.00%,11,"HLA-A*02:01, HLA-A*02:03, HLA-A*23:01, HLA-A*2...",99.33
6,NP 46,GVLESQMLI,9,99.33% (148/149),88.89%,100.00%,11,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",99.33
7,NP 32,TCLWPKTHTL,10,99.33% (148/149),90.00%,100.00%,12,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",99.33
8,NP 54,LWPKTHTLW,9,99.33% (148/149),88.89%,100.00%,10,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*3...",99.33
9,NP 57,TCLWPKTHTLW,11,99.33% (148/149),90.91%,100.00%,10,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",99.33
